# SVO descriptor variant generation

Source: archived SVO notebook, cells 11–17. This reproduces radius-4, 20-point area descriptors. External conformer, log and reported-reaction paths must be configured.


In [ ]:
from pathlib import Path
import os
import sys

def repository_root(start):
    for path in (start, *start.parents):
        if (path / "Code" / "model_train.py").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the CycloAddGenerator repository.")

REPO_ROOT = repository_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks" / "03_modeling"))
DATA_DIR = REPO_ROOT / "Data" / "DFT_Result"
DESCRIPTOR_MAP = REPO_ROOT / "Data" / "Descriptors" / "Descriptors_.pkl"

In [ ]:
import pickle
import numpy as np
import pandas as pd
from rdkit import Chem
from tqdm.auto import tqdm
from Code import cycle_process, xtb_process, logfile_process, main, model_train

SMILES_DIR = None  # External directory with descriptors, all_smiles.csv and mol/.
BO_FILE = None     # External all_reported_BO_One.csv.
RADIUS = 4
NUM_POINTS = 20
OUTPUT_MAP = REPO_ROOT / "Data" / "Descriptors" / "Descriptors_SVO_2.0.pickle"
if SMILES_DIR is None or BO_FILE is None:
    raise ValueError("Set SMILES_DIR and BO_FILE before generating SVO descriptors")
SMILES_DIR, BO_FILE = Path(SMILES_DIR), Path(BO_FILE)
for required in [SMILES_DIR / "Descriptors_appeared.csv",
                 SMILES_DIR / "all_smiles.csv", BO_FILE]:
    if not required.is_file():
        raise FileNotFoundError(required)

In [ ]:
bo_df = pd.read_csv(BO_FILE)
di_df = pd.read_csv(DATA_DIR / "Full_Space.csv")
reactant_ids = (set(bo_df["Diene_Index"]) | set(bo_df["Ene_Index"])
                | set(di_df["Diene_Index"]) | set(di_df["Ene_Index"]))
smiles_table = pd.read_csv(SMILES_DIR / "all_smiles.csv")
smiles_table = smiles_table[smiles_table["Index"].isin(reactant_ids)]
conf_id_by_reactant = dict(zip(smiles_table["Index"],
                               smiles_table["Stable_conf_id"]))
NEW_DESCRIPTORS = model_train.New_des

In [ ]:
descriptor_table = pd.read_csv(SMILES_DIR / "Descriptors_appeared.csv")
descriptor_table = descriptor_table[
    descriptor_table["Smiles_Id"].isin(reactant_ids)
].reset_index(drop=True)
records = []
for _, row in tqdm(descriptor_table.iterrows(), total=len(descriptor_table)):
    smiles_id = int(row["Smiles_Id"])
    title = [int(value) for value in row["Title"].split()]
    mol_file = SMILES_DIR / "mol" / f"smilesid_{smiles_id:05}.mol"
    mol = Chem.MolFromMolFile(str(mol_file), removeHs=False)
    conf_id = int(conf_id_by_reactant[smiles_id])
    log_file = SMILES_DIR / "mol_dft" / f"smilesid_{smiles_id:05}_{conf_id:04}.log"
    log = logfile_process.Logfile(str(log_file))
    mol = xtb_process.xtb_to_mol(mol, [log.symbol_list],
                                 [log.running_positions[-1]], 1)
    result = None
    if row["Type"] == "Diene":
        for positioned, atom_ids, _ in cycle_process.change_position(
                mol, prop="diene", return_tran_cis=True):
            if atom_ids[0] == title[0] and atom_ids[-1] == title[1]:
                result = model_train.Calc_areas(positioned, atom_ids,
                                                radius=RADIUS, num=NUM_POINTS)
                break
        areas = list(result) + [np.nan] * 8 if result is not None else None
    elif row["Type"] == "Ene":
        for positioned, atom_ids in cycle_process.change_position(mol, prop="dieno"):
            if atom_ids[0] == title[0] and atom_ids[-1] == title[1]:
                result = model_train.Calc_areas(positioned, atom_ids,
                                                radius=RADIUS, num=NUM_POINTS)
                break
            if atom_ids[0] == title[1] and atom_ids[-1] == title[0]:
                values = model_train.Calc_areas(positioned, atom_ids,
                                                radius=RADIUS, num=NUM_POINTS)
                result = [values[i] for i in (5, 4, 7, 6, 1, 0, 3, 2)]
                break
        areas = [np.nan] * 8 + list(result) if result is not None else None
    else:
        raise ValueError(f"Unknown reactant type: {row['Type']}")
    if areas is None or len(areas) != 16:
        raise ValueError(f"Could not calculate SVO areas for reactant {smiles_id}")
    records.append(areas)
descriptor_table[NEW_DESCRIPTORS] = np.asarray(records)

In [ ]:
descriptor_map = {}
for _, row in descriptor_table.iterrows():
    smiles_id = int(row["Smiles_Id"])
    if row["Type"] == "Diene":
        type_id, base, area = 1, main.ALL_PROPERTIES_diene, NEW_DESCRIPTORS[:8]
    else:
        type_id, base, area = 0, main.ALL_PROPERTIES_ene, NEW_DESCRIPTORS[8:]
    key = f"{smiles_id:05} {type_id} {row['Title']}"
    descriptor_map[key] = [row[name] for name in base + area]
OUTPUT_MAP.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_MAP.open("wb") as handle:
    pickle.dump(descriptor_map, handle, protocol=pickle.HIGHEST_PROTOCOL)
print(f"Saved {len(descriptor_map):,} reactant-site descriptors to {OUTPUT_MAP}")